# Controlled validation: LRW frontend + BiGRU with tighter crops

This notebook changes only the spatial crop inside the existing landmark v3 96x96 mouth ROI:

- crop 88x88 -> model input 88x88
- crop 80x80 -> resize to 88x88
- crop 72x72 -> resize to 88x88

Everything else is fixed: landmark v3, LRW-pretrained frontend, BiGRU, weighted sampler, speaker split, AdamW, CrossEntropyLoss and seeds 42/123/2026.

This notebook loads only train.csv and val.csv. The test split is intentionally not loaded.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import random
import shutil
import subprocess
import sys
import time
import zipfile

import cv2
import gdown
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

device = 'cuda' if torch.cuda.is_available() else 'cpu'
assert device == 'cuda', 'Enable a GPU runtime before running this notebook.'
print('device:', device)

In [ ]:
# Restore the separate landmark v3 archive. No main dataset is modified.
drive_root = Path('/content/drive/MyDrive/lipreading_sem4')
extract_dir = Path('/content/lipreading_data_tight_crop_v3')
relative_split_dir = Path('03_splits') / 'ml_splits_mouth_crops_landmark_v3' / 'speaker_top10'
data_root = extract_dir / 'ru_dataset'
split_dir = data_root / relative_split_dir

if not split_dir.exists():
    matches = list(Path('/content/drive').rglob('colab_lipreading_dataset_landmark_v3.zip'))
    if not matches:
        raise FileNotFoundError('Upload colab_lipreading_dataset_landmark_v3.zip to Google Drive first.')
    if extract_dir.exists():
        shutil.rmtree(extract_dir)
    extract_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(matches[0], 'r') as archive:
        archive.extractall(extract_dir)
    print('restored:', matches[0])

train_df = pd.read_csv(split_dir / 'train.csv')
val_df = pd.read_csv(split_dir / 'val.csv')
vocab = (split_dir / 'vocab.txt').read_text(encoding='utf-8').splitlines()
word_to_idx = {word: index for index, word in enumerate(vocab)}
idx_to_word = {index: word for word, index in word_to_idx.items()}

assert len(train_df) == 1085
assert len(val_df) == 105
assert set(train_df['speaker_id']).isdisjoint(set(val_df['speaker_id']))
assert all((data_root / path).is_file() for path in train_df['clip_path'])
assert all((data_root / path).is_file() for path in val_df['clip_path'])

output_dir = drive_root / 'controlled_lrw_tight_crop_landmark_v3'
output_dir.mkdir(parents=True, exist_ok=True)
print('train:', len(train_df), sorted(train_df['speaker_id'].unique()))
print('validation:', len(val_df), sorted(val_df['speaker_id'].unique()))
print('vocab:', vocab)
print('test split: not loaded')

## Fixed LRW frontend and variable crop

The landmark videos are 96x96. We take a centered inner crop of 88, 80 or 72 pixels and resize every result to the same 88x88 frontend input. Therefore the frontend sees the same tensor shape in all conditions.

In [ ]:
def set_reproducible_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.use_deterministic_algorithms(True, warn_only=True)

def load_torch_file(path, device='cpu'):
    try:
        return torch.load(path, map_location=device, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=device)

def load_lrw_video(path, inner_crop_size, num_frames=24, input_size=96, model_size=88):
    if inner_crop_size not in (88, 80, 72):
        raise ValueError('inner_crop_size must be 88, 80 or 72')
    cap = cv2.VideoCapture(str(path))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total_frames <= 0:
        cap.release()
        raise ValueError(f'Cannot read video: {path}')
    frames = []
    indices = np.linspace(0, total_frames - 1, num_frames).astype(int)
    offset = (input_size - inner_crop_size) // 2
    for frame_index in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_index))
        ok, frame = cap.read()
        if not ok:
            continue
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        frame = cv2.resize(frame, (input_size, input_size))
        frame = frame[offset:offset + inner_crop_size, offset:offset + inner_crop_size]
        if inner_crop_size != model_size:
            frame = cv2.resize(frame, (model_size, model_size), interpolation=cv2.INTER_LINEAR)
        frame = frame.astype(np.float32) / 255.0
        frames.append((frame - 0.421) / 0.165)
    cap.release()
    if not frames:
        raise ValueError(f'No frames loaded: {path}')
    while len(frames) < num_frames:
        frames.append(frames[-1].copy())
    return torch.from_numpy(np.stack(frames)).float().unsqueeze(0)

class CropVideoDataset(Dataset):
    def __init__(self, metadata, crop_size):
        self.metadata = metadata.reset_index(drop=True)
        self.crop_size = crop_size
    def __len__(self):
        return len(self.metadata)
    def __getitem__(self, index):
        row = self.metadata.iloc[index]
        video = load_lrw_video(data_root / row['clip_path'], self.crop_size)
        return video, index

# Show that all variants have the same model-facing shape.
for crop_size in (88, 80, 72):
    sample = load_lrw_video(data_root / train_df.iloc[0]['clip_path'], crop_size)
    print(f'crop {crop_size}:', tuple(sample.shape))

In [ ]:
# Load the same official LRW checkpoint as in the landmark v3 BiGRU experiment.
upstream_dir = Path('/content/Lipreading_using_Temporal_Convolutional_Networks')
if not upstream_dir.exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/mpc001/Lipreading_using_Temporal_Convolutional_Networks.git', str(upstream_dir)], check=True)
sys.path.insert(0, str(upstream_dir))
from lipreading.model import Lipreading

checkpoint_path = drive_root / 'upstream_lrw_pretrained' / 'lrw_resnet18_mstcn_video.pth'
if not checkpoint_path.exists():
    checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
    result = gdown.download(id='1vqMpxZ5LzJjg50HlZdj_QFJGm2gQmDUD', output=str(checkpoint_path), quiet=False)
    if result is None:
        raise RuntimeError('LRW checkpoint download failed')

tcn_options = {'num_layers': 4, 'kernel_size': [3, 5, 7], 'dropout': 0.2, 'dwpw': False, 'width_mult': 1}
lrw_encoder = Lipreading(
    modality='video', num_classes=500, tcn_options=tcn_options,
    backbone_type='resnet', relu_type='swish', width_mult=1.0,
    use_boundary=False, extract_feats=True,
)
checkpoint = load_torch_file(checkpoint_path)
state_dict = checkpoint.get('model_state_dict', checkpoint)
lrw_encoder.load_state_dict(state_dict, strict=True)
for parameter in lrw_encoder.parameters():
    parameter.requires_grad = False
lrw_encoder = lrw_encoder.to(device).eval()
print('LRW frontend loaded and frozen')

In [ ]:
def extract_features(crop_name, crop_size, metadata, cache_path):
    if cache_path.exists():
        cache = load_torch_file(cache_path)
        print('loaded cache:', cache_path)
        return cache
    loader = DataLoader(CropVideoDataset(metadata, crop_size), batch_size=4, shuffle=False, num_workers=2)
    features = []
    with torch.no_grad():
        for videos, _ in loader:
            videos = videos.to(device)
            batch_features = lrw_encoder(videos, lengths=[videos.shape[2]] * videos.shape[0])
            features.append(batch_features.cpu())
    cache = {
        'features': torch.cat(features),
        'clip_paths': metadata['clip_path'].tolist(),
        'words': metadata['word'].tolist(),
        'speakers': metadata['speaker_id'].tolist(),
        'crop_name': crop_name,
        'inner_crop_size': crop_size,
        'model_input_size': 88,
        'test_used': False,
    }
    torch.save(cache, cache_path)
    print('saved cache:', cache_path)
    return cache

class FeatureSequenceDataset(Dataset):
    def __init__(self, cache):
        self.features = cache['features'].float()
        self.labels = torch.tensor([word_to_idx[word] for word in cache['words']], dtype=torch.long)
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, index):
        return self.features[index], self.labels[index]

class BiGRUClassifier(nn.Module):
    def __init__(self, num_classes, input_dim=512, hidden_dim=128, dropout=0.4):
        super().__init__()
        self.temporal_encoder = nn.GRU(input_dim, hidden_dim, num_layers=1, batch_first=True, bidirectional=True)
        self.classifier = nn.Sequential(nn.Dropout(dropout), nn.Linear(hidden_dim * 2, num_classes))
    def forward(self, features):
        _, hidden = self.temporal_encoder(features)
        clip_features = torch.cat([hidden[-2], hidden[-1]], dim=1)
        return self.classifier(clip_features)

def classification_metrics(preds, labels):
    matrix = np.zeros((len(vocab), len(vocab)), dtype=int)
    for true_index, pred_index in zip(labels, preds):
        matrix[true_index, pred_index] += 1
    rows = []
    for class_id in range(len(vocab)):
        tp = int(matrix[class_id, class_id])
        support = int(matrix[class_id].sum())
        predicted = int(matrix[:, class_id].sum())
        precision = tp / predicted if predicted else 0.0
        recall = tp / support if support else 0.0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
        rows.append({'word': idx_to_word[class_id], 'f1': f1, 'recall': recall})
    return {
        'accuracy': float(np.trace(matrix) / matrix.sum()),
        'macro_f1': float(np.mean([row['f1'] for row in rows])),
        'balanced_accuracy': float(np.mean([row['recall'] for row in rows])),
        'predicted_classes': int(np.count_nonzero(matrix.sum(axis=0))),
        'confusion_matrix': matrix,
    }

In [ ]:
from collections import Counter

def make_loaders(train_cache, val_cache, seed):
    group_keys = list(zip(train_cache['speakers'], train_cache['words']))
    group_counts = Counter(group_keys)
    weights = torch.tensor([1.0 / group_counts[key] for key in group_keys], dtype=torch.double)
    sampler = WeightedRandomSampler(weights, num_samples=len(weights), replacement=True, generator=torch.Generator().manual_seed(seed))
    train_loader = DataLoader(FeatureSequenceDataset(train_cache), batch_size=32, sampler=sampler, num_workers=0)
    val_loader = DataLoader(FeatureSequenceDataset(val_cache), batch_size=32, shuffle=False, num_workers=0)
    return train_loader, val_loader

def evaluate(model, loader, criterion):
    model.eval()
    loss_sum = 0.0
    preds, labels = [], []
    with torch.no_grad():
        for batch_x, batch_y in loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            outputs = model(batch_x)
            loss_sum += criterion(outputs, batch_y).item() * batch_y.size(0)
            preds.extend(outputs.argmax(dim=1).cpu().tolist())
            labels.extend(batch_y.cpu().tolist())
    metrics = classification_metrics(preds, labels)
    metrics['loss'] = loss_sum / len(labels)
    return metrics

def run_one_seed(crop_name, train_cache, val_cache, seed, num_epochs=30, patience=7):
    set_reproducible_seed(seed)
    train_loader, val_loader = make_loaders(train_cache, val_cache, seed)
    model = BiGRUClassifier(len(vocab)).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    best = None
    best_state = None
    stale = 0
    for epoch in range(1, num_epochs + 1):
        model.train()
        for batch_x, batch_y in train_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            optimizer.zero_grad()
            loss = criterion(model(batch_x), batch_y)
            loss.backward()
            optimizer.step()
        metrics = evaluate(model, val_loader, criterion)
        if best is None or metrics['macro_f1'] > best['macro_f1']:
            best = dict(metrics, seed=seed, best_epoch=epoch, crop_name=crop_name)
            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
            stale = 0
        else:
            stale += 1
        if stale >= patience:
            break
    save_dir = output_dir / crop_name
    save_dir.mkdir(parents=True, exist_ok=True)
    torch.save({'model_state_dict': best_state, 'result': best, 'test_used': False}, save_dir / f'seed{seed}_best.pt')
    print(crop_name, 'seed=', seed, 'epoch=', best['best_epoch'], 'acc=', round(best['accuracy'], 4), 'macro-F1=', round(best['macro_f1'], 4), 'balanced_acc=', round(best['balanced_accuracy'], 4))
    return best

In [ ]:
CROP_VARIANTS = {'crop_88': 88, 'crop_80': 80, 'crop_72': 72}
SEEDS = [42, 123, 2026]
all_rows = []

for crop_name, crop_size in CROP_VARIANTS.items():
    variant_dir = output_dir / crop_name
    variant_dir.mkdir(parents=True, exist_ok=True)
    train_cache = extract_features(crop_name, crop_size, train_df, variant_dir / 'train_features.pt')
    val_cache = extract_features(crop_name, crop_size, val_df, variant_dir / 'val_features.pt')
    assert train_cache['features'].shape == (1085, 24, 512)
    assert val_cache['features'].shape == (105, 24, 512)
    for seed in SEEDS:
        all_rows.append(run_one_seed(crop_name, train_cache, val_cache, seed))

runs_df = pd.DataFrame(all_rows)
runs_df.to_csv(output_dir / 'tight_crop_validation_multiseed_runs.csv', index=False)
summary = runs_df.groupby('crop_name')[['accuracy', 'macro_f1', 'balanced_accuracy']].agg(['mean', 'std'])
display(summary)
summary.to_csv(output_dir / 'tight_crop_validation_summary.csv')

fig, ax = plt.subplots(figsize=(7, 4))
means = runs_df.groupby('crop_name')['macro_f1'].mean().reindex(CROP_VARIANTS.keys())
stds = runs_df.groupby('crop_name')['macro_f1'].std().reindex(CROP_VARIANTS.keys())
ax.bar(means.index, means.values, yerr=stds.values, capsize=4)
ax.set_ylabel('validation macro-F1')
ax.set_title('Tighter mouth crop ablation, landmark v3')
fig.tight_layout()
fig.savefig(output_dir / 'tight_crop_validation_macro_f1.png', dpi=160)
plt.show()

best_crop = means.idxmax()
print('Recommended crop by mean validation macro-F1:', best_crop)
print('No test split was loaded or evaluated.')